[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jmdvinodjmd/agentic-ai-tutorial/blob/feature/notebook-rebuild/notebooks/case_studies/data_analysis_assistant/openai_agents.ipynb)

# Data-analysis assistant — OpenAI Agents SDK

**Task.** Produce a supported household-waste summary using typed Analyst and Reporter agents.

**Read this notebook as:** choose a provider → load data → declare the SDK adapter and agents → execute and validate.

In [ ]:
# 1. Choose the model provider; then use Run All. No terminal command is needed.
MODEL_PROVIDER = "mock"  # mock | local | api
MOCK_MODEL_NAME = "analysis-case-v1"
LOCAL_MODEL_NAME = "Qwen3-0.6B-Q8_0"
LOCAL_MODEL_PATH = "models/local/Qwen3-0.6B-Q8_0.gguf"
API_MODEL_NAME = "gemini-3.1-flash-lite"
SAVE_API_CREDENTIAL = True  # saves hidden input to ignored .private/ storage
# Mock runtime is under one minute on CPU; local and API runs can be slower.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "openai-agents==0.17.8", "pydantic==2.12.5"],
        check=True,
    )

In [ ]:
import csv
import json
from pathlib import Path
from typing import ClassVar, Literal

from agents import Agent, Runner, set_tracing_disabled
from agents.items import ModelResponse as SDKResponse
from agents.models.interface import Model as SDKModel
from agents.usage import InputTokensDetails, Usage
from openai.types.responses import ResponseOutputMessage, ResponseOutputText
from pydantic import BaseModel, ConfigDict

set_tracing_disabled(True)
if InputTokensDetails.model_fields["cache_write_tokens"].is_required():
    InputTokensDetails.model_fields["cache_write_tokens"].default = 0
    InputTokensDetails.model_rebuild(force=True)
ROOT = Path.cwd()
while ROOT.parent != ROOT and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists() and "google.colab" in sys.modules:
    ROOT = Path("/content/agentic-ai-tutorial")
    if not ROOT.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "feature/notebook-rebuild",
                "https://github.com/jmdvinodjmd/agentic-ai-tutorial.git",
                str(ROOT),
            ],
            check=True,
        )
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run from the cloned repository.")
sys.path.insert(0, str(ROOT / "src"))
from agentic_tutorial.models import GenerationSettings, create_model  # noqa: E402
from agentic_tutorial.notebook import prepare_gemini_api_key  # noqa: E402
from agentic_tutorial.schemas import Message, MessageRole  # noqa: E402
from agentic_tutorial.tools import AnalysisRequest, file_sha256, summarise_reduction  # noqa: E402

data_path = ROOT / "data/data_analysis_assistant/household_waste.csv"
expected = json.loads((ROOT / "data/data_analysis_assistant/expected_summary.json").read_text())
fixture_path = ROOT / "data/data_analysis_assistant/case_mock.json"
fixture = json.loads(fixture_path.read_text())
if MODEL_PROVIDER == "api":
    prepare_gemini_api_key(ROOT, save=SAVE_API_CREDENTIAL)

## Typed Agents and execution boundary
The analyst may select only a typed operation and columns. No model-authored code is evaluated; deterministic output is checked against the versioned oracle before the reporter receives it.

In [ ]:
# --- Declarations: typed outputs, model adapter, and workflow helpers ---
class Strict(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Decision(Strict):
    schema_id: ClassVar[str] = "analysis.decision.v1"
    tool: Literal["mean_reduction_by_intervention"]
    group_column: Literal["intervention"]
    before_column: Literal["before_kg"]
    after_column: Literal["after_kg"]


class Report(Strict):
    schema_id: ClassVar[str] = "analysis.report.v1"
    report: str
    groups: tuple[str, ...]


Decision.model_rebuild(_types_namespace={"Literal": Literal})


def core_model():
    model_names = {"mock": MOCK_MODEL_NAME, "local": LOCAL_MODEL_NAME, "api": API_MODEL_NAME}
    model_path = ROOT / LOCAL_MODEL_PATH if MODEL_PROVIDER == "local" else None
    return create_model(
        provider=MODEL_PROVIDER,
        mock_fixture_path=str(fixture_path),
        model=model_names[MODEL_PROVIDER],
        model_path=model_path,
        metadata_path=ROOT / "models/local/model_metadata.json",
        settings=GenerationSettings(temperature=0.0, max_output_tokens=1024, seed=0),
        options={"timeout_seconds": 180.0},
    )


def user(text):
    return Message(role=MessageRole.USER, content=text)


class FixtureModel(SDKModel):
    def __init__(self):
        self.core = core_model()

    async def get_response(
        self,
        system_instructions,
        input,
        model_settings,
        tools,
        output_schema,
        handoffs,
        tracing,
        *,
        previous_response_id,
        conversation_id,
        prompt,
    ):
        schema = None if output_schema is None else output_schema.output_type
        request = f"{system_instructions}\n\n{input}"
        response = await self.core.generate([user(request)], response_schema=schema)
        item = ResponseOutputMessage(
            id=response.response_id,
            content=[
                ResponseOutputText(
                    annotations=[],
                    text=json.dumps(response.structured_output, sort_keys=True),
                    type="output_text",
                    logprobs=[],
                )
            ],
            role="assistant",
            status="completed",
            type="message",
        )
        return SDKResponse(output=[item], usage=Usage(), response_id=response.response_id)

    async def stream_response(self, *args, **kwargs):
        if False:
            yield None


async def run_analysis():
    model = FixtureModel()
    trace = []
    with data_path.open(newline="", encoding="utf-8") as handle:
        rows = list(csv.DictReader(handle))
    columns = tuple(rows[0])
    trace.append({"event": "inspect", "rows": len(rows)})
    analyst = Agent(
        name="Analyst",
        instructions="Select one permitted operation and existing columns; never write code.",
        model=model,
        output_type=Decision,
    )
    decision = (
        await Runner.run(
            analyst,
            f"Columns: {columns}. Select mean_reduction_by_intervention with "
            "intervention, before_kg and after_kg.",
            max_turns=2,
        )
    ).final_output
    trace.append({"event": "decision", "tool": decision.tool})
    if decision.tool != "mean_reduction_by_intervention":
        return {"outcome": "fallback", "trace": trace}
    request = AnalysisRequest(
        group_column=decision.group_column,
        before_column=decision.before_column,
        after_column=decision.after_column,
    )
    result = summarise_reduction(data_path, request)
    valid = result == expected["mean_reduction_kg"]
    trace.append({"event": "execute_and_validate", "valid": valid})
    if not valid:
        return {"outcome": "fallback", "trace": trace}
    reporter = Agent(
        name="Reporter",
        instructions="Report only validated supplied values.",
        model=model,
        output_type=Report,
    )
    report = (
        await Runner.run(
            reporter, f"Validated results: {result}. Include every key once in groups.", max_turns=2
        )
    ).final_output
    groups_valid = set(report.groups) == set(result)
    trace.append({"event": "report_validation", "valid": groups_valid})
    return {
        "decision": decision,
        "result": result,
        "report": report,
        "outcome": "supported_report" if groups_valid else "fallback",
        "trace": trace,
        "model_calls": 2,
        "termination": "criteria_met" if groups_valid else "validation_failed",
    }


# --- Execution: run the workflow and evaluate its observable result ---
first = await run_analysis()
second = await run_analysis() if MODEL_PROVIDER == "mock" else first
try:
    AnalysisRequest(group_column="missing", before_column="before_kg", after_column="after_kg")
    schema_mismatch_rejected = False
except ValueError:
    schema_mismatch_rejected = True
failure_demonstrations = {
    "invalid_tool": "blocked before deterministic tool",
    "unsafe_code": "never evaluated",
    "schema_mismatch": schema_mismatch_rejected,
    "result_mismatch": "report Agent not run",
}
evaluation = {
    "component": first["result"] == expected["mean_reduction_kg"],
    "trajectory": len(first["trace"]) == 4 and first["model_calls"] <= 2,
    "task": first["outcome"] == "supported_report",
    "safety": first["decision"].tool == "mean_reduction_by_intervention"
    and schema_mismatch_rejected,
    "repeated_run": first == second,
}
if MODEL_PROVIDER == "mock":
    assert all(evaluation.values()), evaluation
{
    "evaluation": evaluation,
    "trace": first["trace"],
    "resource_report": {"model_calls": first["model_calls"], "agents": 2},
    "failure_demonstrations": failure_demonstrations,
    "provenance": file_sha256(data_path),
    "fallback": "invalid typed output stops before report",
}

## Limitation
Typed Agents constrain orchestration but do not make the small descriptive dataset causal; arbitrary code is excluded.